# Functional Capstone Project - Data & Analytics



## Business Context

**Infini Telco** is a prominent player in the telco industry with a strong presence in Malaysia. Specializing in B2B sales, they serve as a vital link in the supply chain, facilitating the distribution of telecommunication and technology products to retailers, service providers, and other businesses across the continent.

Despite their market dominance and expansive product offerings, the company faces critical business challenge related to marketing strategy. A key aspect of this involves the ***identification and segmentation of their customer base into well-defined segments based on discernible patterns in product purchase behaviour and understand their product portfolio.*** By understanding the distinct needs, preferences, and purchasing habits of different customer segments, the client can develop targeted marketing strategies that cater to the specific needs of each segment, enhancing customer satisfaction and loyalty, driving revenue growth.




## Business Challenges

- No streamlined process to handle data and data is only available in chunks
- Limited intel on customer purchase patterns to drive targeted marketing
- Inadequate product insights across different customer groups


## Project Objectives

The business needs your help as a ***data analyst*** to overcome these challenges. Apply your data analytics skillset learnt.



### 1. Data Preprocessing

Data preparation is a critical process first step, involving the gathering, cleaning, transforming, and organizing of raw data into a format suitable for analysis. This step is essential for ensuring the quality and reliability of the insights derived from the data.

Here, the transaction data is provided from June 2020 to December 2022, which are stored across three separate csv files. The dataset are:

- Transactions_1.csv
- Transactions_2.csv
- Transactions_3.csv

***Task 1.1: Combine the transaction data files***

Find a way to combine the above data together into a singular dataframe, that will contain all the transactions at one place. Name this singular dataframe df_Txn_full.

In [ ]:
#Import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import warnings
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
warnings.filterwarnings("ignore")

pd.options.display.float_format = '{:.2f}'.format
pd.options.display.max_rows = 100

In [ ]:
# Import transaction files
df_Txn_1 = pd.read_csv('Transactions_1.csv')
df_Txn_2 = pd.read_csv('Transactions_2.csv')
df_Txn_3 = pd.read_csv('Transactions_3.csv')


In [ ]:
# Combine transaction datasets
df_Txn_full = pd.concat([df_Txn_1, df_Txn_2, df_Txn_3], ignore_index=True)

# Preview data
df_Txn_full.head()


In [ ]:
# Check dataframe structure
print(df_Txn_full.shape)
print(df_Txn_full.columns)
df_Txn_full.info()


**Validation checkpoints:**
Consider, you would have received some information from the business on:

- Gross Annual Turnover for this period to be ~$1.4B
- The total unique products are ~190000
- The total customer base is ~7k

***Task 1.1.1***:

Use the above information to validate the data creation process. This is to ensure that no records of data are lost in the process.

In [ ]:
# Validation checks
gross_turnover = df_Txn_full['Gross_Turnover'].sum()
total_products = df_Txn_full['Product'].nunique()
total_customers = df_Txn_full['customer_number'].nunique()

print(f'Gross Annual Turnover: {gross_turnover:,.2f}')
print(f'Total Unique Products: {total_products}')
print(f'Total Customer Base: {total_customers}')


***Task 1.2: Join the mapping files to the transaction data***


Mapping files contain supplementary detailed information that will be used for our analysis.
Use these mapping files to join with the above transaction data to create the master dataset <br><br>
**List of mapping files and their primary keys:**<br>
1. Customer Characteristics : to be joined on Customer Number
2. Customer Type Mapping : to be joined on Main Account ID
3. Product Category Mapping : to be joined on Product ID

To ensure that the joins are performed accurately, it is a good practice to clean up the columns upon which the join is performed.
Some recommended steps to clean up a column are:

- ensuring the values are of the same data type
- removing leading zeroes
- removing leading and trailing whitespaces etc.


***Task 1.2.1:***

Here, before, performing the joins, you have to perform an intermediary step to clean up the below columns based on the above recommended steps:

- customer_number
- Product
- Main_account
- Invoice_Number

*Hint: Check for any leading or trailing 0s or white spaces.*

In [ ]:
# Clean join columns
join_cols = ['customer_number', 'Product', 'Main_account', 'Invoice_Number']

for col in join_cols:
    if col in df_Txn_full.columns:
        df_Txn_full[col] = (
            df_Txn_full[col]
            .astype(str)
            .str.strip()
            .str.lstrip('0')
        )

df_Txn_full.head()


In [ ]:
# Additional cleanup
df_Txn_full.columns = df_Txn_full.columns.str.strip()

print('Data cleaned successfully')


***Task 1.2.2:***

Import the mapping files and rename the dataframe:

- Customer Characteristics.csv : df_CustChar
- Customer Type Mapping.csv : df_CustType
- Product Categories.csv : df_ProdCategories

In [ ]:
# Load mapping files
df_CustChar = pd.read_csv('Customer Characteristics.csv')
df_CustType = pd.read_csv('Customer Type Mapping.csv')
df_ProdCategories = pd.read_csv('Product Categories.csv')

print('Mapping files imported successfully')


In [ ]:
# Preview mapping files
print(df_CustChar.head())
print(df_CustType.head())
print(df_ProdCategories.head())


***Task 1.2.3:***

To perform join on the dataframes, it is important to clean the columns to ensure that the columns are free from any special characters.

- Remove any leading or trailing 0s in Customer_Number column in the df_CustChar dataframe.
- Check for duplicates in the data and remove any duplicates.
- Create a master dataframe by performing appropriate join on df_CustChar dataframe and df_Txn_full dataframe. Rename the master dataframe as df_Master.



In [ ]:
# Clean Customer Characteristics dataframe
df_CustChar['Customer_Number'] = (
    df_CustChar['Customer_Number']
    .astype(str)
    .str.strip()
    .str.lstrip('0')
)

df_CustChar = df_CustChar.drop_duplicates()

# Merge transaction + customer characteristics
df_Master = df_Txn_full.merge(
    df_CustChar,
    left_on='customer_number',
    right_on='Customer_Number',
    how='left'
)

print(df_Master.shape)


***Task 1.2.4:***

There is a data validation check point here. This is to ensure that you have not lost any data in the data preparation step.
Check and validate:

- Gross Annual Turnover is ~$1.4B
- The total customer base is ~7k

In [ ]:
# Validation after first merge
print(f"Gross Annual Turnover: {df_Master['Gross_Turnover'].sum():,.2f}")
print(f"Total Customer Base: {df_Master['customer_number'].nunique()}")


***Task 1.2.5:***

Now, you will be joining the df_CustType dataframe with the master dataframe.

- Remove any leading or trailing 0s in Main_account_ID column in the df_CustType dataframe.
- Check for duplicates in the data and remove any duplicates.
- Perform appropriate join on df_CustType dataframe and df_Master dataframe and save this merged dataframe as df_Master.

In [ ]:
# Clean Customer Type Mapping
df_CustType['Main_account_ID'] = (
    df_CustType['Main_account_ID']
    .astype(str)
    .str.strip()
    .str.lstrip('0')
)

df_CustType = df_CustType.drop_duplicates()

# Merge customer type mapping
df_Master = df_Master.merge(
    df_CustType,
    left_on='Main_account',
    right_on='Main_account_ID',
    how='left'
)

print(df_Master.shape)


In [ ]:
# Preview merged dataframe
df_Master.head()


***Task 1.2.6:***

There is a data validation check point here. This is to ensure that you have not lost any data in the data preparation step.
Check and validate:

- Gross Annual Turnover is ~$1.4B
- The total customer base is ~7k

In [ ]:
# Validation after second merge
print(f"Gross Annual Turnover: {df_Master['Gross_Turnover'].sum():,.2f}")
print(f"Total Customer Base: {df_Master['customer_number'].nunique()}")


***Task 1.2.7:***

Now, you will be joining the df_ProdCategories dataframe with the master dataframe.

- Remove any leading or trailing 0s in ProductID column in the df_ProdCategories dataframe.
- Check for duplicates in the data and remove any duplicates.
- Perform appropriate join on df_ProdCategories dataframe and df_Master dataframe and save this merged dataframe as df_Master.

In [ ]:
# Clean Product Categories dataframe
df_ProdCategories['ProductID'] = (
    df_ProdCategories['ProductID']
    .astype(str)
    .str.strip()
    .str.lstrip('0')
)

df_ProdCategories = df_ProdCategories.drop_duplicates()

# Merge product categories
df_Master = df_Master.merge(
    df_ProdCategories,
    left_on='Product',
    right_on='ProductID',
    how='left'
)

print(df_Master.shape)


***Task 1.2.8:***

There is a data validation check point here. This is to ensure that you have not lost any data in the data preparation step.
Check and validate:

- Gross Annual Turnover is ~$1.4B
- The total customer base is ~7k
- The total product is ~15k

In [ ]:
# Final validation
print(f"Gross Annual Turnover: {df_Master['Gross_Turnover'].sum():,.2f}")
print(f"Total Customer Base: {df_Master['customer_number'].nunique()}")
print(f"Total Products: {df_Master['Product'].nunique()}")


***Task 1.3: Data Cleaning***

A major part of data pre processing involves cleaning the data, removing nulls, filtering out irrelavant or less useful information. For example:

- Remove rows with missing product id
- Removing rows with negative or zero turnover
- Removing customers who interacted very less

As part of Data Cleaning of this dataset you are required to perform the following:

***Task 1.3.1: In the df_Master data, add a column Invoice_flag, reflecting 0 if number of transactions <= 3, otherwise 1***

This is to flag customers who purchase less frequently

In [ ]:
# Create Invoice_flag
invoice_counts = df_Master.groupby('customer_number')['Invoice_Number'].transform('nunique')

df_Master['Invoice_flag'] = np.where(invoice_counts <= 3, 0, 1)

df_Master[['customer_number', 'Invoice_flag']].head()


***Task 1.3.2: In the df_Master data, add a column Invoice_flag, reflecting 0 if number of transactions <= 3, otherwise 1***

This is to flag customers who purchase single SKU.

In [ ]:
# Create SKU_flag
sku_counts = df_Master.groupby('customer_number')['Product'].transform('nunique')

df_Master['SKU_flag'] = np.where(sku_counts <= 1, 0, 1)

df_Master[['customer_number', 'SKU_flag']].head()


There are a few records in the dataset whose Product ID are null. These records won't map to product information from the mapping file and hence these records will not be useful for the analysis.

***Task 1.3.3: In the master data, remove records where Product ID is null and save it in a dataframe df_Filtered***

In [ ]:
# Remove records with null Product ID
df_Filtered = df_Master[df_Master['Product'].notna()].copy()

print(df_Filtered.shape)


There are a few records that lack information on product category and will not be useful for the analysis

***Task 1.3.4: In the df_Filtered dataftame, remove records where Product Category information is null***


In [ ]:
# Remove records with null Product Category
if 'Product_Category' in df_Filtered.columns:
    df_Filtered = df_Filtered[df_Filtered['Product_Category'].notna()]
elif 'Product_Category_Name' in df_Filtered.columns:
    df_Filtered = df_Filtered[df_Filtered['Product_Category_Name'].notna()]

print(df_Filtered.shape)


There are some sales made to Internal accounts in the dataset. These sales can be removed from analysis as information on internal sales can distort the analysis of customer behaviour. Internal transactions do not reflect the actual market demand or customer preferences.

***Task 1.3.5: In the df_Filtered dataframe, remove records pertaining to sales to Internal Account***

*Hint: Check for Internal accounts in Account_Group_TXT field*

In [ ]:
# Remove internal account sales
df_Filtered = df_Filtered[
    ~df_Filtered['Account_Group_TXT']
    .astype(str)
    .str.contains('Internal', case=False, na=False)
]

print(df_Filtered.shape)


As per business requirement, sales made to "ZSKA" and "ZDIR" are requested to be removed.

***Task 1.3.6: In the df_Filtered dataframe, remove records with position types "ZSKA" and "ZDIR"***

*Hint: Check column Position_type_order_line*

In [ ]:
# Remove ZSKA and ZDIR position types
df_Filtered = df_Filtered[
    ~df_Filtered['Position_type_order_line'].isin(['ZSKA', 'ZDIR'])
]

print(df_Filtered.shape)


Some companies are currently not active or are in normal operation. It makes sense to exclude these companies from analysis. But, for companies whose status is unknown (null/na), we still retain them.

***Task 1.3.7: In the df_Filtered dataframe, remove records whose company status are not active/normal***

*Hint: Check column CVR_Company_status*

In [ ]:
# Keep only active/normal company status
df_Filtered = df_Filtered[
    (df_Filtered['CVR_Company_status'].isna()) |
    (df_Filtered['CVR_Company_status']
        .astype(str)
        .str.lower()
        .isin(['active', 'normal']))
]

print(df_Filtered.shape)


There might be some records where Gross Turnover is zero or negative. These records might indicate product returns/invalid entries and should be excluded from the analysis.

***Task 1.3.8: In the df_Filtered dataframe, remove records where Gross Turnover is zero or negative***

In [ ]:
# Remove zero or negative gross turnover
df_Filtered = df_Filtered[df_Filtered['Gross_Turnover'] > 0]

print(df_Filtered.shape)


There might be some records where Amount is zero or negative. These records might indicate product returns/invalid entries and should be excluded from the analysis.

***Task 1.3.9: In the df_Filtered dataframe, remove records where Amount is zero or negative***


In [ ]:
# Remove zero or negative amount
df_Filtered = df_Filtered[df_Filtered['Amount'] > 0]

print(df_Filtered.shape)


Remember, you had created a column to flag the customers whose number of transactions are 3 or less than that. Customers with very few transactions may introduce noise into the data, making it harder to identify meaningful patterns and trends. Filtering out these customers results in a cleaner dataset, which enhances the accuracy and reliability of the analysis.

***Task 1.3.10: In the df_Filtered dataframe, filter out customers who have made 3 or fewer transactions***

*Hint: Utilize the Invoice_flag column created earlier.*

In [ ]:
# Filter customers with more than 3 invoices
df_Filtered = df_Filtered[df_Filtered['Invoice_flag'] == 1]

print(df_Filtered.shape)


You had also created a column to flag the customers who purchased only one sku. Such records in the data also tends to add noise and is necesary to filter these customers out.

***Task 1.3.11: In the df_Filtered dataframe, filter out customers who have purchased only a single SKU***

*Hint: Utilize the SKU_flag column created earlier.*

In [ ]:
# Filter customers with more than 1 SKU
df_Filtered = df_Filtered[df_Filtered['SKU_flag'] == 1]

print(df_Filtered.shape)


There are some information from the business and the business aligned on the fact that some very large customers are outliers and having them in the data could skew the analysis results. Hence, they should be filtered out. These customer types are denoted by KAM (Key Account Management).

***Task 1.3.12: In the df_Filtered dataframe, Filter out "Key" customer accounts***

*Hint: Check Customer_type column*

In [ ]:
# Remove Key customer accounts
df_Filtered = df_Filtered[
    ~df_Filtered['Customer_type']
    .astype(str)
    .str.contains('Key', case=False, na=False)
]

print(df_Filtered.shape)


**With the above steps of data preprocessing, you now get a cleaned data set which will be used for further analysis.**

### 2. Data Transformation


***Task 2.1: Based on what you have learnt previously, perform a series of standard EDA to gain a better understanding of the data***

In [ ]:
# Basic EDA
print(df_Filtered.describe(include='all'))

# Missing values
print(df_Filtered.isnull().sum())

# Top product categories
if 'Product_Category' in df_Filtered.columns:
    print(df_Filtered['Product_Category'].value_counts().head(10))

# Revenue distribution
plt.figure(figsize=(10,6))
plt.hist(df_Filtered['Gross_Turnover'], bins=50)
plt.xlabel('Gross Turnover')
plt.ylabel('Frequency')
plt.title('Distribution of Gross Turnover')
plt.show()

# Correlation matrix
numeric_cols = df_Filtered.select_dtypes(include=np.number)

plt.figure(figsize=(10,8))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## Pareto Analysis

**Pareto Analysis** is a strategic decision-making tool used to prioritize actions based on the principle that a small number of causes typically lead to a large portion of the effects. Known as the 80/20 rule or the Pareto Principle, this concept is highly applicable in customer and product management, helping businesses focus on the most impactful areas to maximize efficiency and profitability.

## Key Concepts


**i) Pareto Principle (80/20 Rule):**

In the context of customers and products, this principle suggests that roughly 80% of a company’s revenue comes from 20% of its customers or products. Understanding this can help businesses focus their efforts on the most valuable segments.

**ii) Pareto Chart:**

A Pareto chart is a graphical tool that helps visualize and prioritize the most significant factors among a set. It combines a bar graph and a line graph, where bars represent individual values in descending order, and the line shows the cumulative total.

***An example of Pareto Chart and Pareto Analysis:***

![image-2.png](attachment:image-2.png)

Here is the Pareto chart for products and their corresponding revenues. The chart consists of two key elements:

**Bar Chart:** The blue bars represent the individual revenue contributions of each product, sorted in descending order.

**Line Chart:** The red line shows the cumulative percentage of the total revenue contributed by the products. The horizontal
gray dashed line indicates the 80% mark.

***Analysis***

- Product Contribution: The first few products (A, B, and C) generate a significant portion of the revenue, highlighting the "vital few" that follow the Pareto Principle.
- Cumulative Percentage: By the time we include Product E, the cumulative revenue reaches around 80% of the total, illustrating that a small number of products contribute to the majority of the revenue.

***Implications***

- Focus on Key Products: Efforts should be concentrated on optimizing and promoting these top-performing products to maximize revenue.
- Resource Allocation: Inventory, marketing, and development resources should be prioritized for these high-impact products to enhance business efficiency and profitability.
- Customer Strategies: Understanding which products are most valued by customers can help in tailoring marketing and customer engagement strategies.


Using Pareto analysis helps businesses identify and focus on the most impactful products, ensuring efficient use of resources and targeted efforts towards driving revenue growth.

To execute the pareto analysis, you must create the required dataset. There are two levels of Product category in this dataset, use ProdCat2 for this analysis.

***Task 2.2: Create a new dataframe df_Pareto with the columns: ProdCat2, Gross_Turnover, Cumulative_GT, Cumulative_GT%***

*Hint: Cumulative_GT is Cumulative Gross Turnover which you can get by doing a cumulative sum on the Gross Turnover.*



In [ ]:
#Insert your code

In [ ]:
# #Insert your code

***Task 2.3: Arrive at a "Pareto filtered" dataset df_Pareto_80 with the top product categories that contribute to 80% of the Gross Turnover***

*Hint: There are 64 products that contribute to 80% of Gross Revenue turnover*

In [ ]:
#Insert your code

***Task 2.3: Display the Pareto Chart***

*Hint: Your Pareto Chart will look something like this*

![image.png](attachment:image.png)


***Task 2.4: Perform the necessary action to map the top products from Pareto Analysis to the previously created master dataframe df_Filtered and store this in a new dataframe df_Filtered_Pareto.***

*Hint: Think Joins?*

In [ ]:
# Input codes here

In [ ]:
# Input codes here

In the above dataset, you will notice that there are some columns that are repititive and redundant.

***Task 2.5: Remove the redundant columns and give a meaningful name to the repititive columns***

In [ ]:
#Insert your code

**This analysis gives an understanding of the top product portfolio of the company. This will be further used to perform customer segmentation.**

### 3. Customer Segmentation

The next part in this analysis is to find meaningful customer segments from the product purchase behaviour.

You will use **KMeans Clustering Algorithm** for this.

### KMeans Algorithm
The KMeans algorithm is a popular clustering technique used to partition a dataset into K clusters, where each data point belongs to the cluster with the nearest mean. Here are the steps involved in the KMeans algorithm:

**Initialization:**

- Choose the optimal number of clusters K, say 3 in this case.
- Randomly select K data points from the dataset as the initial centroids (cluster centers).

![image-2.png](attachment:image-2.png)

**Assignment Step:**

- Assign each data point to the nearest centroid. This is usually done by calculating the Euclidean distance between each data point and the centroids.
- Each data point is assigned to the cluster whose centroid is closest to it.

**Update Step:**

- Recalculate the centroids as the mean of all data points assigned to each cluster.
- The new centroid for each cluster is the average of the positions of all the data points in that cluster.

![image-3.png](attachment:image-3.png)

**Repeat:**

- Repeat the Assignment and Update steps until the centroids no longer change significantly or a specified number of iterations is reached.
- This convergence indicates that the clusters are stable, and the algorithm has found the optimal clustering.

![image-4.png](attachment:image-4.png)

**Termination:**

- The algorithm terminates when the centroids have stabilized (i.e., they do not change significantly between iterations) or after a pre-defined number of iterations.



To execute this analysis, **Main Account** and **ProdCat1** (is a broader product category and is at the highest level in the product hierarchy) data will be used.

*Note: ProdCat2 level data is at a very granular level and performing clustering analysis on this might not give meaningful clusters.*

To proceed with the customer segmentation analysis, you need to menaingfully prepare the data.

***Task 3.1: Transform the data to feed to the clustering algorithm***

- Create a copy of the above dataframe and save it as **df_segment**
- Create a Primary Key: Concatenate the Main Account (ID) and Main Account Name
- Execute a groupby operation to calculate the annual turnover of each customer for each product (ProdCat1)
- Calculate the % spend by each customer across different product categories

The resultant dataframe will have the following columns:
- Primary Key
- ProdCat1
- Gross_Turnover
- Customer Spend %

In [ ]:
#Insert your code

***Task 3.2: Pivot the dataset to create a matrix view of spend % of all customers on all products (ProdCat1). Save the dataframe as df_pivot. Create another dataframe df_clustering removing the primary key from the df_pivot dataframe.***

In [ ]:
#Insert your code

### The Elbow Method in KMeans Clustering

The Elbow Method is a commonly used technique to determine the optimal number of clusters (K) in KMeans clustering. It helps to balance between underfitting and overfitting by finding a point where adding more clusters doesn’t significantly improve the model performance.

![image.png](attachment:image.png)

***Explanation of the Plot***

**WCSS (Within Cluster Sum of Squares) vs. Number of Clusters:**

- The x-axis represents the number of clusters (K).
- The y-axis represents the Within-Cluster Sum of Squares (WCSS), which measures the variance within each cluster.

**Plotting WCSS for Different K Values:**

- For each value of K (from 1 to 10), the KMeans algorithm is run, and the corresponding WCSS is calculated.
- The plot shows how WCSS decreases as the number of clusters increases.

**Elbow Point:**

- The red point marked on the plot indicates the "elbow point" where the rate of decrease in WCSS slows down.
- In this example, the elbow point is at K = 3. This suggests that 3 clusters are optimal for this dataset, as adding more clusters beyond this point results in only a marginal reduction in WCSS.

**Interpretation:**

- Before the Elbow Point: Adding more clusters significantly reduces WCSS, indicating that the clusters are becoming more defined and compact.
- After the Elbow Point: The reduction in WCSS slows down, meaning that adding more clusters doesn’t significantly improve the clustering.

**Conclusion**

The Elbow Method helps to identify the optimal number of clusters by looking for the point where the WCSS starts to decrease at a slower rate, balancing model complexity and performance. In this case, the plot suggests that using 3 clusters is a good choice for the dataset.

***Task 3.3: Run K-Means clustering algorithm and identify the optimal number of clusters using Elbow Method***

In [ ]:
#Insert your code

The Elbow Curve provides an optimal value for 'k' that is the number of clusters to be generated in the output. This value can be used as a guideline for the actual clustering run; one may vary the number of clusers in the output based on business need.

***Task 3.4: Run the KMeans algorithm with optimal number of clusters***

In [ ]:
#Insert your code

***Task 3.5: From the above clusters, create cluster profile***

***Task 3.5.1: Map each customers to their respective cluster in df_segment dataset***

In [ ]:
#Insert your code

***Task 3.5.2: Create a cluster profile table with the cluster label, Gross Turnover for each cluster, Number of unique customers in each cluster, % share of Gross Turnover of each cluster for each product.***

*Illustrative: Here is a snapshot of the desired cluster profile:*
![image-2.png](attachment:image-2.png)

In [ ]:
#Insert your code

### 4. Prepare data for Tableau Dashboard

From here, you will be creating the visualizations in Tableau. But before, creating the required visualizations in Tableau, you should prepare the data.

Here is a reference of how data for visualizations for **Customer Segmentation** looks like:

![image-5.png](attachment:image-5.png)






Here is a reference of how data for visualizations for **Product Categorization** looks like:

![image-4.png](attachment:image-4.png)

where:

- Product ID: SKU ID
- Gross Turnover (sum), Amount (sum), Invoice_count (distinct count of invoices), Main_account_count (distinct count of main accounts): fields aggregated at SKU level
- Above columns with _percluster: fields aggregated at cluster level
- Frequency (%) : number of transactions the SKU appeared on, when compared to all transactions in a cluster
- Customer Prevalence (%) within cluster : Percentage of customers within a cluster who purchase the SKU



In [ ]:
# Insert your codes here

In [ ]:
# Insert your codes here

In [ ]:
# Insert your codes here

In [ ]:
# Insert your codes here

In [ ]:
# Insert your codes here

***The End***

## 2. Data Transformation

Create customer-level aggregated metrics for segmentation analysis.

In [ ]:
# Convert invoice date to datetime if available
date_col = None
for col in df_Filtered.columns:
    if 'date' in col.lower():
        date_col = col
        break

if date_col:
    df_Filtered[date_col] = pd.to_datetime(df_Filtered[date_col], errors='coerce')

print(f'Date column used: {date_col}')

In [ ]:
# Create customer-level summary dataset
customer_summary = df_Filtered.groupby('customer_number').agg({
    'Gross_Turnover': 'sum',
    'Amount': 'sum',
    'Invoice_Number': 'nunique',
    'Product': 'nunique'
}).reset_index()

customer_summary.columns = [
    'customer_number',
    'Total_Revenue',
    'Total_Amount',
    'Total_Invoices',
    'Unique_Products'
]

customer_summary.head()

In [ ]:
# Create average order value
customer_summary['Avg_Order_Value'] = (
    customer_summary['Total_Revenue'] /
    customer_summary['Total_Invoices']
)

# Create product diversity ratio
customer_summary['Product_Diversity'] = (
    customer_summary['Unique_Products'] /
    customer_summary['Total_Invoices']
)

customer_summary.describe()

## 3. Customer Segmentation

Apply clustering techniques to identify customer groups.

In [ ]:
# Features for segmentation
seg_features = customer_summary[[
    'Total_Revenue',
    'Total_Amount',
    'Total_Invoices',
    'Unique_Products',
    'Avg_Order_Value',
    'Product_Diversity'
]]

# Standardize data
scaler = preprocessing.StandardScaler()
scaled_features = scaler.fit_transform(seg_features)

print('Scaling completed')

In [ ]:
# Elbow method
inertia = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K, inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
# Silhouette score analysis
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(scaled_features)
    score = silhouette_score(scaled_features, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8,5))
plt.plot(range(2,11), silhouette_scores, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.show()

best_k = range(2,11)[np.argmax(silhouette_scores)]
print(f'Best cluster count based on silhouette score: {best_k}')

In [ ]:
# Final KMeans clustering
kmeans = KMeans(n_clusters=best_k, random_state=42)
customer_summary['Cluster'] = kmeans.fit_predict(scaled_features)

customer_summary.head()

In [ ]:
# Cluster profiling
cluster_profile = customer_summary.groupby('Cluster').agg({
    'Total_Revenue': 'mean',
    'Total_Amount': 'mean',
    'Total_Invoices': 'mean',
    'Unique_Products': 'mean',
    'Avg_Order_Value': 'mean'
}).round(2)

cluster_profile

In [ ]:
# Visualize clusters
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=customer_summary,
    x='Total_Revenue',
    y='Total_Invoices',
    hue='Cluster'
)
plt.title('Customer Segments')
plt.show()

## 4. Prepare Data for Tableau Dashboard

Export cleaned and transformed datasets for Tableau dashboard development.

In [ ]:
# Export datasets
df_Filtered.to_csv('Cleaned_Transaction_Data.csv', index=False)
customer_summary.to_csv('Customer_Segmentation_Data.csv', index=False)
cluster_profile.to_csv('Cluster_Profile_Summary.csv')

print('CSV files exported successfully')

In [ ]:
# Suggested Tableau KPIs
total_revenue = customer_summary['Total_Revenue'].sum()
total_customers = customer_summary['customer_number'].nunique()
total_invoices = customer_summary['Total_Invoices'].sum()
avg_order_value = customer_summary['Avg_Order_Value'].mean()

print(f'Total Revenue: {total_revenue:,.2f}')
print(f'Total Customers: {total_customers}')
print(f'Total Invoices: {total_invoices}')
print(f'Average Order Value: {avg_order_value:,.2f}')